# C5-neural-networks — Session 2: MLPs and Geometry

*One class session, roughly 85 minutes. Prerequisites: Session 1 (the
perceptron, the step activation, why activations are necessary) and
F2-vectors (lines, sides of a line, the dot product).*

**This session:** wiring perceptrons into a **multi-layer perceptron (MLP)**
and making it compute geometry on purpose.
The plan: architecture vocabulary and the two NumPy helpers this course pins
(`affine_layer`, `step_activation`), a full forward pass by hand, then the
design pattern the exam loves — each hidden unit detects a **half-plane**,
a second-layer unit **ANDs** the detectors, and suddenly the network answers
"is this point inside the region?" for any polygon you care to describe.
Every weight in this session is designed, not trained (Session 1's scope
note).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804

## 1. From One Perceptron to a Network

**Definition.**
A **multi-layer perceptron (MLP)** is a chain of **layers**.
Each layer holds several units side by side; every unit computes Session 1's
recipe — an affine pre-activation, then an activation — reading the
*previous layer's outputs* as its input.
The layers between input and output are **hidden layers**, their units
**hidden units**; the number of units in a layer is its **width**, the
number of layers the **depth**.

**Architecture notation, pinned:** we write $2 \to 4 \to 1$ for a network
taking 2 inputs, passing them through one hidden layer of 4 units, and
producing 1 output.
As a function it is a **composition**: with $f^{(1)}$ the hidden layer
(affine map plus activation) and $f^{(2)}$ the output layer,

$$y = f^{(2)}\!\big(f^{(1)}(x)\big).$$

Nothing more mysterious than functions feeding functions — but Session 1
§5 says the composition only earns its keep because of the activation
between the affine maps.

**Shape bookkeeping (memorize this paragraph).**
A layer from $d_{\text{in}}$ inputs to $d_{\text{out}}$ units carries a
weight matrix $W$ of shape $(d_{\text{out}}, d_{\text{in}})$ — **row $j$
holds the weights of unit $j$** — and a bias vector $b$ of shape
$(d_{\text{out}},)$.
Data comes in batches: $x$ of shape $(n, d_{\text{in}})$, one point per
row, exactly as in C2.

### Checkpoint 1

1. For a $3 \to 5 \to 2$ network: how many hidden units are there, and
   what are the shapes of both layers' $W$ and $b$?
2. Write the $2 \to 3 \to 1$ forward computation as a composition of named
   functions, marking where the activation acts.
3. Which Session 1 fact says the hidden activation cannot be dropped
   without collapsing the network? One sentence.

## 2. The Two Pinned Helpers: `affine_layer` and `step_activation`

The whole unit — and unit C6-pytorch after it, by these exact names — runs
on two NumPy functions.
Their contracts are fixed; the exam register states contracts like these and
grades against them.

**`affine_layer(x, W, b)`** — the pre-activations of one layer, in the
component form of this course:

$$z_{ij} \;=\; \sum_{k} x_{ik}\, W_{jk} + b_j ,$$

for `x` of shape $(n, d_{\text{in}})$, `W` of shape
$(d_{\text{out}}, d_{\text{in}})$, `b` of shape $(d_{\text{out}},)$,
returning shape $(n, d_{\text{out}})$: entry $(i, j)$ is unit $j$'s
pre-activation on point $i$.

**`step_activation(z)`** — Session 1's threshold, elementwise, returning
floats.

Both are written broadcasting-only (F1's idiom, C2's register — no `@`, no
`np.dot`, no transposes, no loops):

In [ ]:
def affine_layer(x, W, b):
    """Pre-activations z[i, j] = sum_k x[i, k] * W[j, k] + b[j].

    x: (n, d_in)   W: (d_out, d_in)   b: (d_out,)   ->   (n, d_out)
    """
    return (x[:, None, :] * W[None, :, :]).sum(axis=2) + b


def step_activation(z):
    """Threshold activation, elementwise: 1.0 where z >= 0, else 0.0."""
    return (z >= 0).astype(float)


# Session 1's worked perceptron, now as a width-1 layer:
W = np.array([[2.0, -1.0]])          # (1, 2): one unit, two inputs
b = np.array([-3.0])
pts = np.array([[2.0, 0.0], [1.0, 1.0], [0.0, 3.0], [3.0, 3.0]])
print("z :", affine_layer(pts, W, b)[:, 0])
print("out:", step_activation(affine_layer(pts, W, b))[:, 0])

Same numbers as Session 1 §1 — $z = (1, -2, -6, 0)$, outputs
$(1, 0, 0, 1)$ — but the code now scales to any width and any batch.

How the broadcast implements the component sum: `x[:, None, :]` has shape
$(n, 1, d_{\text{in}})$ and `W[None, :, :]` has shape
$(1, d_{\text{out}}, d_{\text{in}})$; the product is
$(n, d_{\text{out}}, d_{\text{in}})$ holding every $x_{ik} W_{jk}$, and
`.sum(axis=2)` collapses the shared $k$ axis.
The `+ b` broadcasts across rows.

### Checkpoint 2

1. `x` has shape $(7, 3)$ and the layer is $3 \to 4$.
   What are the shapes of `W`, `b`, and `affine_layer(x, W, b)`?
2. By hand: for `x = [[1, 2]]`, `W = [[3, -1], [0, 2]]`, `b = (1, -5)`,
   compute both entries of `affine_layer(x, W, b)`.
3. Why does `affine_layer` need no loop over units? One sentence about
   which broadcast axis plays that role.

## 3. A Full Forward Pass, by Hand and by NumPy

**The network:** $2 \to 2 \to 1$, step activation at the hidden layer *and*
at the output (a yes/no machine end to end):

$$W^{(1)} = \begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix},
\quad b^{(1)} = (-0.5,\; -2.5), \qquad
W^{(2)} = \begin{pmatrix} 1 & 1 \end{pmatrix},
\quad b^{(2)} = (-1.5).$$

Hidden unit 1 fires when $x_1 - x_2 \ge 0.5$; hidden unit 2 fires when
$x_1 + x_2 \ge 2.5$.

**By hand, point $(2, 1)$:**

| stage | computation | result |
|---|---|---|
| $z^{(1)}$ | $(2 - 1 - 0.5,\;\; 2 + 1 - 2.5)$ | $(0.5,\; 0.5)$ |
| $h$ | $\mathrm{step}$ of each | $(1,\; 1)$ |
| $z^{(2)}$ | $1 + 1 - 1.5$ | $0.5$ |
| $y$ | $\mathrm{step}(0.5)$ | $1$ |

**By hand, point $(0, 1)$:**
$z^{(1)} = (0 - 1 - 0.5,\; 0 + 1 - 2.5) = (-1.5, -1.5)$, so $h = (0, 0)$,
$z^{(2)} = -1.5$, $y = 0$.

The same chain in code is three composed calls:

In [ ]:
W1 = np.array([[1.0, -1.0], [1.0, 1.0]])
b1 = np.array([-0.5, -2.5])
W2 = np.array([[1.0, 1.0]])
b2 = np.array([-1.5])

x = np.array([[2.0, 1.0], [0.0, 1.0]])
h = step_activation(affine_layer(x, W1, b1))
y = step_activation(affine_layer(h, W2, b2))
print("hidden h:")
print(h)
print("output y:", y[:, 0])

The prints agree with both hand tables: $h$ rows $(1, 1)$ and $(0, 0)$,
outputs $(1, 0)$.
Notice what the output unit computed: with both hidden outputs in
$\{0, 1\}$, the pre-activation $h_1 + h_2 - 1.5$ is $\ge 0$ exactly when
*both* detectors fire — an AND.
That observation is the whole next act.

### Checkpoint 3

1. Forward the point $(3, 0)$ through this network by hand: $z^{(1)}$,
   $h$, $z^{(2)}$, $y$.
2. Change $b^{(2)}$ to $-0.5$ and forward $(0, 3)$.
   What does the output unit now compute, in logic words?
3. Point $(1.5, 1.0)$: which hidden pre-activation lands exactly on 0, and
   what does the convention say happens?

## 4. One Hidden Unit = One Half-Plane Detector

**The geometric reading.**
A step unit with weights $w = (w_1, w_2)$ and bias $b$ fires on

$$\{\,x : w_1 x_1 + w_2 x_2 + b \ge 0\,\},$$

which is a **half-plane**: everything on one side of the straight line
$w_1 x_1 + w_2 x_2 + b = 0$, boundary included.
Which side? The pre-activation *grows* in the direction of $w$ — so $w$
points **into** the firing side.
(F2: $w$ is normal to the line, and moving along $w$ increases the dot
product.)

**Reading a detector, worked.**
Hidden unit 1 above has $w = (1, -1)$, $b = -0.5$: boundary
$x_1 - x_2 = 0.5$, a $45°$ line; $w = (1, -1)$ points right-and-down, so
the unit fires on the lower-right side — where $x_1$ is large and $x_2$
small.

**Writing a detector, worked (the design direction).**
Want a unit that fires exactly on $x_2 \le 2$.
Rewrite as $\ge 0$: $\;2 - x_2 \ge 0$, so $w = (0, -1)$, $b = 2$.
*Always rewrite the target inequality into the pinned form
$w \cdot x + b \ge 0$ first; the coefficients then read off.*

In [ ]:
rng = np.random.default_rng(SEED)
cloud = rng.uniform([-3, -3], [4, 4], (400, 2))
det = step_activation(affine_layer(cloud, np.array([[1.0, -1.0]]), np.array([-0.5])))[:, 0]

plt.figure(figsize=(5.2, 4.4))
plt.scatter(cloud[det == 1, 0], cloud[det == 1, 1], s=9, label="fires")
plt.scatter(cloud[det == 0, 0], cloud[det == 0, 1], s=9, label="silent")
xs = np.linspace(-3, 4, 50)
plt.plot(xs, xs - 0.5, color="k", linewidth=1, label="$x_1 - x_2 = 0.5$")
plt.arrow(0.5, 0.0, 0.5, -0.5, head_width=0.15, color="k")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("A half-plane detector; the arrow is $w$, pointing into the firing side")
plt.legend(loc="upper left", fontsize=8)
plt.show()

### Checkpoint 4

1. Write $(w, b)$ for detectors of: (i) $x_1 \ge -1$; (ii) $x_2 \ge x_1$;
   (iii) $x_1 + 2 x_2 \le 4$.
2. For the detector $w = (2, 1)$, $b = -4$: does $(1, 2)$ fire? Does
   $(1, 1)$? Show the pre-activations.
3. Two detectors share the same boundary line but fire on opposite sides.
   How are their $(w, b)$ related, and on which points do they *both*
   fire?

## 5. Logic on Detectors: AND, OR, NOT

A second-layer unit reads $m$ step outputs $h \in \{0, 1\}^m$.
With all weights $1$, its pre-activation is the **count of firing
detectors** $S = \sum_j h_j$ plus the bias — so choosing the bias turns the
unit into a logic gate:

| gate | weights | bias | fires iff |
|---|---|---|---|
| AND of $m$ | all $1$ | $-(m - 0.5)$ | $S \ge m - 0.5 \iff S = m$ (all fire) |
| OR of $m$ | all $1$ | $-0.5$ | $S \ge 0.5 \iff S \ge 1$ (any fires) |
| NOT | $-1$ | $0.5$ | $0.5 - h \ge 0 \iff h = 0$ |
| at least $r$ of $m$ | all $1$ | $-(r - 0.5)$ | $S \ge r$ |

**Why the half-integer biases:** $S$ is a whole number, so thresholds at
half-integers leave a safety margin of $\tfrac12$ on both sides.
(With exact $0/1$ floats, $-m$ would also work for AND — the margin is
what keeps designs robust when detector outputs are ever so slightly off,
and it is the register this course pins.)
The four input combinations of a 2-detector AND and OR, checked
exhaustively:

In [ ]:
corners = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])   # all h-patterns
and_gate = step_activation(affine_layer(corners, np.array([[1.0, 1.0]]), np.array([-1.5])))[:, 0]
or_gate = step_activation(affine_layer(corners, np.array([[1.0, 1.0]]), np.array([-0.5])))[:, 0]
print("h pattern   AND   OR")
for hpat, a, o in zip(corners, and_gate, or_gate):
    print(f"{hpat}   {a:.0f}    {o:.0f}")

The printed truth table is the familiar one: AND fires only on
$(1, 1)$; OR fires on all but $(0, 0)$.

### Checkpoint 5

1. Design the second-layer unit "at least 2 of 3 detectors fire".
2. Design "NOT detector 1, AND detector 2" as a single unit reading
   $(h_1, h_2)$ — weights and bias.
3. Why does no *single* unit reading $(h_1, h_2)$ compute "exactly one
   fires" (XOR)? Argue from $S$-thresholds in one sentence.

## 6. Polygon Membership by Half-Plane AND-ing

**The design pattern, in full.**
A convex polygon is an intersection of half-planes.
So a $2 \to m \to 1$ step network answers "inside?" exactly:

1. write each edge as an inequality $w \cdot x + b \ge 0$ with the polygon
   on the $\ge$ side — one hidden detector per edge;
2. AND the $m$ detectors with the output unit (weights $1$, bias
   $-(m - 0.5)$).

Boundary included: on an edge the detector reads $z = 0$ and fires, so
edges and corners count as inside — the pinned $\ge$ convention doing real
work.

**Worked region.**
The quadrilateral $Q$ cut out by

$$x_1 \ge 0, \qquad x_2 \ge 0, \qquad x_1 + x_2 \le 6, \qquad x_2 \le x_1 + 3,$$

with corners $(0,0)$, $(6,0)$, $(1.5, 4.5)$, $(0, 3)$.
Rewriting the last two as $\ge 0$: $\;6 - x_1 - x_2 \ge 0$ and
$x_1 - x_2 + 3 \ge 0$.
Stacking all four into one hidden layer (one row per edge, in the order
stated):

$$W^{(1)} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ -1 & -1 \\ 1 & -1 \end{pmatrix},
\quad b^{(1)} = (0,\; 0,\; 6,\; 3), \qquad
W^{(2)} = (1\;\; 1\;\; 1\;\; 1), \quad b^{(2)} = -3.5 .$$

In [ ]:
W1_q = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0], [1.0, -1.0]])
b1_q = np.array([0.0, 0.0, 6.0, 3.0])
W2_q = np.ones((1, 4))
b2_q = np.array([-3.5])


def inside_Q(points):
    h = step_activation(affine_layer(points, W1_q, b1_q))
    return step_activation(affine_layer(h, W2_q, b2_q))[:, 0]


# corners are inside (boundary fires), a clearly-outside point is not:
probes = np.array([[0.0, 0.0], [6.0, 0.0], [1.5, 4.5], [0.0, 3.0], [5.0, 4.0]])
print("probe labels:", inside_Q(probes))

rng = np.random.default_rng(SEED)
pts = rng.uniform([-1, -1], [7, 6], (4000, 2))
lab = inside_Q(pts)
print("fraction inside:", lab.mean(), "   (exact area ratio 15.75 / 56 = 0.28125)")

plt.figure(figsize=(5.4, 4.4))
plt.scatter(pts[lab == 1, 0], pts[lab == 1, 1], s=6, label="inside Q")
plt.scatter(pts[lab == 0, 0], pts[lab == 0, 1], s=6, alpha=0.35, label="outside")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Polygon membership from a 2 -> 4 -> 1 step network")
plt.legend(loc="upper right", fontsize=8)
plt.show()

All four corners print 1 (edges belong to $Q$), the outside probe prints
0, and the sampled fraction `0.28025` sits within sampling noise of the
exact area ratio $15.75 / 56 = 0.28125$ (the quadrilateral's area by the
shoelace formula, over the sampling box's $8 \times 7$).

Non-convex regions are one idea away: build one AND unit per convex piece
in a second hidden layer, then OR the pieces at the output — a
$2 \to m \to (\text{pieces}) \to 1$ network.
Practice p14 walks it in full.

### Checkpoint 6

1. Add the constraint $x_2 \le 4$ to $Q$: give the new hidden row of
   $W^{(1)}$, the new $b^{(1)}$ entry, and the new output bias.
2. Is $(2, 4)$ in $Q$? Decide from the four inequalities by hand, then
   name which detector(s) a network evaluation would show silent.
3. Why does this construction need the polygon to be *convex*? One
   sentence.

## 7. Worked Exam-Style Example 2: Constrained Coding

The register: exact identifiers, exact shapes, an API ban list with a
zero-points clause.
Solved in full below.

---

**Problem.**
Write `band_indicator(points)` for the band of the plane between the lines
$x_2 = 2 x_1 - 4$ and $x_2 = 2 x_1 + 2$, **both boundary lines included**:
it takes `points` of shape $(n, 2)$ and returns shape $(n,)$ floats —
`1.0` for points in the band, `0.0` otherwise.
Build it as a step network using `affine_layer` and `step_activation`
only.
**Banned (zero points): `@`, `np.matmul`, `np.dot`, `np.einsum`,
`np.tensordot`, `.T`, loops.**

---

**Solution.**

*Step 1 — the band as two $\ge 0$ inequalities.*
Below-or-on the upper line: $2 x_1 + 2 - x_2 \ge 0$, i.e.
$w = (2, -1)$, $b = 2$.
Above-or-on the lower line: $x_2 - 2 x_1 + 4 \ge 0$, i.e.
$w = (-2, 1)$, $b = 4$.

*Step 2 — AND the two detectors.* Output weights $(1, 1)$, bias $-1.5$.

*Step 3 — implement inside the bans* (the helpers already comply):

In [ ]:
W1_band = np.array([[2.0, -1.0], [-2.0, 1.0]])
b1_band = np.array([2.0, 4.0])


def band_indicator(points):
    h = step_activation(affine_layer(points, W1_band, b1_band))
    return step_activation(affine_layer(h, np.ones((1, 2)), np.array([-1.5])))[:, 0]


# probes: center of the band, above it, and exactly on the lower boundary
probes = np.array([[0.0, 0.0], [0.0, 5.0], [1.0, -2.0]])
print("probe labels:", band_indicator(probes))

*Step 4 — verify the probes by hand.*
$(0, 0)$: upper-line detector $2 \cdot 0 + 2 - 0 = 2 \ge 0$ fires;
lower-line detector $0 - 0 + 4 = 4 \ge 0$ fires; AND fires → `1.0`.
$(0, 5)$: upper detector $2 - 5 = -3 < 0$ silent → `0.0`.
$(1, -2)$: lower detector $-2 - 2 + 4 = 0$ — exactly on the boundary
line, fires by the $\ge$ convention; upper fires too ($2 + 2 + 2 = 6$)
→ `1.0`.
The print `[1. 0. 1.]` confirms all three.

### Checkpoint 7

1. Rewrite the band with *both boundaries excluded*.
   Why can no single step network with the pinned $\ge$ convention do it —
   and what pair of NOT-ed detectors does?
2. Widen the band to the lines $x_2 = 2x_1 \pm 5$: which entries of
   $(W^{(1)}, b^{(1)})$ change and to what?

## 8. Common Pitfalls II

**Pitfall 4 — $W$ orientation: $(d_{\text{out}}, d_{\text{in}})$, rows are
units.**
With non-square layers the wrong orientation crashes loudly — good.
The danger is **square** layers, where the transposed matrix runs without
complaint and computes a *different network*:

In [ ]:
x_sq = np.array([[1.0, 2.0]])
W_sq = np.array([[1.0, -1.0], [0.0, 3.0]])
b_sq = np.array([0.0, 0.0])

print("correct rows-are-units:", affine_layer(x_sq, W_sq, b_sq)[0])
print("transposed by mistake :", affine_layer(x_sq, np.array([[1.0, 0.0], [-1.0, 3.0]]), b_sq)[0])

try:
    affine_layer(np.ones((5, 3)), np.ones((3, 4)), np.zeros(4))   # BROKEN shape: W must be (4, 3)
except ValueError as e:
    print("non-square wrong orientation raises:", str(e)[:60], "...")

Read the prints: the correct rows-are-units call gives $(-1, 6)$; the
transposed matrix runs just as happily and gives $(1, 5)$ — a different
network, no warning anywhere.
Keep *row $j$ = unit $j$* sacred, and predict shapes before running
(Checkpoint 2.1's habit); only non-square mistakes announce themselves with
the broadcast error shown.

**Pitfall 5 — dropping the hidden activation.**
Feed pre-activations $z^{(1)}$ straight into the output layer and Session 1
§5 collapses the network: the quadrilateral machine of Section 6 degrades
into a *single straight line* — it cannot represent $Q$ at all:

In [ ]:
def inside_Q_broken(points):
    z1 = affine_layer(points, W1_q, b1_q)                     # BROKEN: no step here
    return step_activation(affine_layer(z1, W2_q, b2_q))[:, 0]


lab_broken = inside_Q_broken(pts)
print("broken 'inside' fraction:", lab_broken.mean(), "   (true region: 0.28025 of this cloud)")
# the broken network is step(sum of all four z's) = one half-plane test:
w_eff = W1_q.sum(axis=0)
print("collapsed detector: w_eff =", w_eff, ", b_eff =", b1_q.sum() + b2_q[0],
      "-> fires iff", f"{w_eff[0]:.0f}*x1 + {w_eff[1]:.0f}*x2 + {b1_q.sum() + b2_q[0]:.1f} >= 0")

The broken classifier fires on `0.97875` of the cloud — nearly the whole
box, nothing like the quadrilateral: with the four hidden steps gone, the
output unit reads $\sum_j z^{(1)}_j - 3.5 = x_1 - x_2 + 5.5$, one tilted
half-plane that misses only the box's top-left sliver.
Symptom to remember: if a "polygon" region comes out as one straight
boundary, an activation went missing.

**Pitfall 6 — gate biases that forget the count is discrete.**
The AND bias must sit *between* $m - 1$ and $m$.
Common slips: $-(m + 0.5)$ can never fire ($S \le m < m + 0.5$);
$-(m - 1)$ already fires when one detector is silent ($S = m - 1$
gives $z = 0$, which fires under $\ge$):

In [ ]:
three_of_four = np.array([[1.0, 1.0, 1.0, 0.0]])
all_four = np.array([[1.0, 1.0, 1.0, 1.0]])
for bias, name in ((-3.5, "correct  -(m-0.5)"), (-4.5, "too deep -(m+0.5)"), (-3.0, "too shallow -(m-1)")):
    g3 = step_activation(affine_layer(three_of_four, np.ones((1, 4)), np.array([bias])))[0, 0]
    g4 = step_activation(affine_layer(all_four, np.ones((1, 4)), np.array([bias])))[0, 0]
    print(f"{name}:  fires on 3-of-4: {g3:.0f}   fires on 4-of-4: {g4:.0f}")

Only the pinned $-3.5$ row behaves as AND — the too-deep bias rejects even
a unanimous count, and the too-shallow one accepts 3-of-4 because $z = 0$
fires.

### Checkpoint 8

1. A teammate's $3 \to 3 \to 1$ region network runs without error but
   classifies a triangle's interior as a half-plane. Two candidate
   pitfalls from this session — which prints would distinguish them?
2. An AND-of-5 unit fires on points where only 4 detectors fire.
   What bias was probably used, and what is the fix?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. 5 hidden units. Layer 1: $W$ is $(5, 3)$, $b$ is $(5,)$; layer 2:
   $W$ is $(2, 5)$, $b$ is $(2,)$.
2. $y = f^{(2)}(f^{(1)}(x))$ with
   $f^{(1)}(x) = \mathrm{act}(W^{(1)} \text{-affine of } x)$ — the
   activation acts on the hidden pre-activations (and on the output too, if
   the design calls for a yes/no output).
3. Session 1 §5: without the activation the two affine maps compose into
   one affine map ($W^{\mathrm{eff}}, b^{\mathrm{eff}}$) — the hidden layer
   would add nothing.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `W`: $(4, 3)$; `b`: $(4,)$; result: $(7, 4)$.
2. Unit 1: $1 \cdot 3 + 2 \cdot (-1) + 1 = 2$; unit 2:
   $1 \cdot 0 + 2 \cdot 2 - 5 = -1$. Result $[[2, -1]]$.
3. The broadcast axis of length $d_{\text{out}}$ (the middle axis of the
   product) enumerates the units — every unit's weighted sum is computed
   simultaneously.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $z^{(1)} = (3 - 0 - 0.5,\; 3 + 0 - 2.5) = (2.5, 0.5)$; $h = (1, 1)$;
   $z^{(2)} = 0.5$; $y = 1$.
2. $z^{(1)} = (0 - 3 - 0.5,\; 0 + 3 - 2.5) = (-3.5, 0.5)$, $h = (0, 1)$,
   $z^{(2)} = 1 - 0.5 = 0.5 \ge 0$, $y = 1$: with bias $-0.5$ the output
   unit computes OR — it fires when *at least one* detector fires.
3. $z^{(1)}_1 = 1.5 - 1.0 - 0.5 = 0$ — exactly on the boundary; the
   pinned $\ge$ convention makes the unit fire ($h_1 = 1$).

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. (i) $w = (1, 0)$, $b = 1$. (ii) $x_2 - x_1 \ge 0$: $w = (-1, 1)$,
   $b = 0$. (iii) $4 - x_1 - 2x_2 \ge 0$: $w = (-1, -2)$, $b = 4$.
2. $(1, 2)$: $z = 2 + 2 - 4 = 0$ — fires (boundary). $(1, 1)$:
   $z = 2 + 1 - 4 = -1$ — silent.
3. Negate both: $(w, b) \to (-w, -b)$. Both fire exactly on the boundary
   line, where both read $z = 0$.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Weights $(1, 1, 1)$, bias $-1.5$: fires iff $S \ge 1.5$, i.e.
   $S \ge 2$.
2. Weights $(-1, 1)$, bias $-0.5$: pre-activation $h_2 - h_1 - 0.5$,
   which is $\ge 0$ only for $(h_1, h_2) = (0, 1)$.
3. A single unit fires on $S$-patterns lying on one side of a threshold,
   but XOR must accept $S = 1$ while rejecting *both* $S = 0$ and $S = 2$
   — no single interval of the form "count $\ge$ threshold" (or its
   negation) does that with equal weights, and unequal weights still give
   one half-plane over $(h_1, h_2)$, which cannot separate the diagonal
   pairs.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. New row $(0, -1)$ with bias entry $4$ (from $4 - x_2 \ge 0$); the
   output unit now ANDs $m = 5$ detectors, bias $-4.5$.
2. Check: $2 \ge 0$ ✓; $4 \ge 0$ ✓; $2 + 4 = 6 \le 6$ ✓ (boundary);
   $4 \le 2 + 3 = 5$ ✓. Inside — in fact on the third edge, and a network
   evaluation shows all four detectors firing (the third at $z = 0$).
3. Because AND-ing half-planes yields exactly the intersections of
   half-planes, and those are precisely the convex regions — a non-convex
   region is not such an intersection (it needs an OR of pieces).

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Strict inequalities exclude the boundary, but every step detector fires
   *on* its boundary ($z = 0$ under $\ge$) — no AND of them can exclude
   it. NOT-ed detectors do it: "strictly below the upper line" is NOT
   ("on or above the upper line"), i.e. NOT of the detector
   $w = (-2, 1)$, $b = -2$ (which fires iff $x_2 \ge 2x_1 + 2$); similarly
   for the lower line with NOT of $w = (2, -1)$, $b = -4$.
   AND the two NOT-ed units in the usual way.
2. Only the biases: $b^{(1)} = (5, 5)$ (from $2x_1 + 5 - x_2 \ge 0$ and
   $x_2 - 2x_1 + 5 \ge 0$). $W^{(1)}$ and the output unit are unchanged.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Missing hidden activation (Pitfall 5) or transposed square $W^{(1)}$
   (Pitfall 4). Distinguish by printing the hidden values: floats far
   outside $\{0, 1\}$ mean the activation is missing; clean $0/1$ patterns
   that disagree with hand-computed detectors mean the orientation is
   wrong.
2. Probably $-(m - 1) = -4$ (or shallower): $S = 4$ then gives $z = 0$,
   which fires under $\ge$. Fix: bias $-4.5 = -(5 - 0.5)$.

</details>